## Import Dataset

In [1]:
# Impot libraries
import pandas as pd

In [2]:
df = pd.read_excel('./datasets/WRMD_2016_to_2024_all_cols.xlsx')
# Using the file above will have more columns that are needed to join on the WCV data
# df = pd.read_csv('WRMD_VA_2016-2025_Records.csv') 

In [3]:
df.columns

Index(['admissions.case_year', 'admissions.hash', 'admissions.id', 'exams.age',
       'exams.age_unit', 'exams.attitude', 'exams.bcs', 'exams.body',
       'exams.cardiopulmonary', 'exams.cns', 'exams.comments',
       'exams.dehydration', 'exams.examined_at', 'exams.examiner',
       'exams.forelimb', 'exams.gastrointestinal', 'exams.head',
       'exams.hindlimb', 'exams.integument', 'exams.mm_color',
       'exams.mm_texture', 'exams.musculoskeletal', 'exams.nutrition',
       'exams.sex', 'exams.temperature', 'exams.temperature_unit',
       'exams.treatment', 'exams.type', 'exams.weight', 'exams.weight_unit',
       'patient_locations.area', 'patient_locations.comments',
       'patient_locations.enclosure', 'patient_locations.moved_in_at',
       'patient_locations.where_holding', 'patients.address_found',
       'patients.admitted_at', 'patients.admitted_by', 'patients.band',
       'patients.carcass_saved', 'patients.care_by_rescuer',
       'patients.city_found', 'patients.cl

In [4]:
df.shape

(25100, 95)

## Drop Rows where there is no geolocation information

In [6]:
# Profile the relevant columns
# df[['patients.address_found', 'patients.county_found', 'patients.city_found']]

In [5]:
#Profile the address_found column
df['patients.address_found'].value_counts()

patients.address_found
ss                                                                                                                                 294
00                                                                                                                                 129
unknown                                                                                                                            124
X                                                                                                                                  120
x                                                                                                                                   91
                                                                                                                                  ... 
776 Old Charles Town Rd.                                                                                                             1
Intersection of Harpers Ferry Rd

Some of the data in 'patients.address_found' is not useful for programatically determining lat long coordinates. We want to drop the rows where the address_found starts with a alphabetic character AND either lat_found or lng_found is null

In [6]:
# Determine How many rows have the 'patients.address_found' column starting with a non-numeric character
df[df['patients.address_found'].str[0].str.isnumeric() == False].shape

(5985, 95)

In [7]:
# Of the rows that have the 'patients.address_found' column starting with a non-numeric character, how many have a null for eithe rlat_found or long_found
df_address_profiling = df[df['patients.address_found'].str[0].str.isnumeric() == False]
df_address_profiling[['patients.lat_found', 'patients.lng_found']].isnull().sum()

patients.lat_found    3965
patients.lng_found    3965
dtype: int64

In [8]:
# determine which rows have the 'patients.address_found' column starting with a non-numeric character 
# df[df['patients.address_found'].str[0].str.isnumeric() == True]

# Set the 'patients.address_found' column to null for these rows
df.loc[df['patients.address_found'].str[0].str.isnumeric() == False, 'patients.address_found'] = None

In [9]:
# Find rows where the 'patients.address_found' column contains only numbers
df[df['patients.address_found'].str.isnumeric() == True]

,admissions.case_year,admissions.hash,admissions.id,exams.age,exams.age_unit,exams.attitude,exams.bcs,exams.body,exams.cardiopulmonary,exams.cns,...,people.notes,people.organization,people.phone,people.postal_code,people.subdivision,species.class,species.family,species.genus,species.order,species.species
1503,2016,NaN,1504,0.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,VA,Reptilia,Emydidae,Terrapene,Testudines,Carolina
1504,2016,NaN,1505,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,VA,Aves,Mimidae,Mimus,Passeriformes,polyglottos
1505,2016,NaN,1506,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,VA,Aves,Passeridae,Passer,Passeriformes,domesticus
1506,2016,NaN,1507,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,VA,Mammalia,Procyonidae,Procyon,Carnivora,lotor
1507,2016,NaN,1508,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,VA,Aves,Passeridae,Passer,Passeriformes,domesticus
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1842,2017,NaN,202,0.0,Juvenile,Quiet,Reasonable,NaN,NaN,"standing on intake, but weak and quickly sank ...",...,Transported by JoDee Grierber from Pender to B...,NaN,NaN,NaN,VA,Aves,Accipitridae,Buteo,Accipitriformes,jamaicensis
2280,2017,NaN,640,0.0,Juvenile,Quiet,Reasonable,NaN,NaN,NaN,...,Wildlife Rescue League,NaN,7034085976,22202,VA,Aves,Accipitridae,Buteo,Accipitriformes,jamaicensis
2803,2017,NaN,1163,0.0,Juvenile,Alert,Reasonable,NaN,NaN,NaN,...,Former Director of Wildlife Services at BRWC,Wildlife Vet Care,540-664-9494,22646,VA,Aves,Accipitridae,Buteo,Accipitriformes,lineatus
3027,2017,NaN,1387,0.0,Adult,Alert,Reasonable,NaN,"open-mouth breathing, lungs auscult clear",NaN,...,NaN,Shenandoah Animal Control,NaN,NaN,VA,Aves,Accipitridae,Accipiter,Accipitriformes,cooperii


In [10]:
# For rows where the 'patients.address_found' column contains only numbers, set the 'patients.address_found' column to null
df.loc[df['patients.address_found'].str.isnumeric() == True, 'patients.address_found'] = None

If we have no address information whatsoever, then drop that row:

In [11]:
# Drop rows where all the following columns are empty: 'patients.address_found', 'patients.lat_found', 'patients.long_found'
df = df.dropna(subset=['patients.address_found', 'patients.lat_found', 'patients.lng_found'], how='all')

In [12]:
df.shape

(21109, 95)

In [13]:
# Find how many rows have a null for either the patients.lat_found or patients.lng_found columns
df[['patients.lat_found', 'patients.lng_found']].isnull().sum()

patients.lat_found    15418
patients.lng_found    15417
dtype: int64

Check if we have incomplete coordinates for any of the rows (only one of lat or long):

In [14]:
# Find rows where only one of patients.lat_found or patients.lng_found columns are null
df[df['patients.lat_found'].isnull() ^ df['patients.lng_found'].isnull()]

,admissions.case_year,admissions.hash,admissions.id,exams.age,exams.age_unit,exams.attitude,exams.bcs,exams.body,exams.cardiopulmonary,exams.cns,...,people.notes,people.organization,people.phone,people.postal_code,people.subdivision,species.class,species.family,species.genus,species.order,species.species
7634,2019,NaN,1972,NaN,Infant,Alert,Reasonable,NaN,NaN,NaN,...,NaN,NaN,7033973526,22309,VA,Mammalia,Sciuridae,Sciurus,Rodentia,carolinensis


In [15]:
# If only one of patients.lat_found or patients.lng_found columns are null, set the other to null
df.loc[df['patients.lat_found'].isnull() ^ df['patients.lng_found'].isnull(), ['patients.lat_found', 'patients.lng_found']] = None

In [ ]:
# Note that some of the entries in the 'patients.address_found' column are not valid addresses.
# Search for the word "mile" within the string to see examples of these entries. Roughly 50

In [16]:
df['patients.address_found'].value_counts()

patients.address_found
106 Island Farm Ln         75
106 Island Farm Lane       65
7503 Cedar Knolls Drive    18
32 Jenkins Ct              16
3307 Halfway Rd            16
                           ..
45871 Debhill Terrace       1
67 Cedar Mountain           1
13115 Hill Club Ln          1
9 Rosepetal St              1
415 5th St                  1
Name: count, Length: 13640, dtype: int64

## Determine Which ones are Vehicle Collision Related

* The patients.keyword column contains relevant information. Search for "HBV" (Hit by Vehicle).
* The patients.diagnosis column also has relevant data - "HBV" and "HBV" as a string in long from text
* the patient.reasons_for_admission column has relevent info: "HBV" among others


### Keyword Column

In [18]:
# Find all rows in the patient.keywords column that contain the string "HBV" and examine the results
vc = df[df['patients.keywords'].str.contains('HBV', regex=False, case=False, na=False)]['patients.keywords'].value_counts()
print(vc)

patients.keywords
suspect HBV                                                553
B - Baby, orphan, mother HBV                               122
B - Baby, mother HBV                                        72
B - Baby, mother HBV, orphan                                43
B - Baby, small, mother HBV                                 24
                                                          ... 
suspect HBV,  suspect cat attack                             1
gunshot, suspect HBV                                         1
suspect HBV, rabies suspect                                  1
B - Baby, mother killed, dog attack, mother HBV, orphan      1
grounded, trauma, suspect HBV                                1
Name: count, Length: 76, dtype: int64


In [19]:
# Grab the indexes of the rows that contain the string "HBV" in the patient.keywords column
keyword_column_hbv = df[df['patients.keywords'].str.contains('HBV', regex=False, case=False, na=False)].index

### Diagnosis Column

In [20]:
df['patients.diagnosis'].value_counts()

patients.diagnosis
T - Unknown Trauma            4072
B - Baby                      3577
D - Domestic animal attack    3550
M - Human non-intentional     1234
H - Hit by Vehicle            1173
                              ... 
T - Unknown traum                1
T - trauma                       1
predator attack                  1
t                                1
B - Baby, B - Baby               1
Name: count, Length: 114, dtype: int64

In [21]:
# Find all rows in the patient.diagnosis column that are equal to "H - Hit by Vehicle" and examine the results
df[df['patients.diagnosis'].str.contains('Vehicle', regex=False, case=False, na=False)]['patients.diagnosis'].value_counts()

patients.diagnosis
H - Hit by Vehicle     1173
H - Hit by vehicle      233
H- Hit by vehicle        84
H - Hit by vehicle       39
H- hit by vehicle         1
H -Hit by vehicle         1
H - hit by vehicle        1
Name: count, dtype: int64

In [22]:
# Grab the indexes of the rows that contain the string "Vehicle" in the patient.diagnosis column
diagnosis_column_vehicle = df[df['patients.diagnosis'].str.contains('Vehicle', regex=False, case=False, na=False)].index


### Reasons for Admission Column

In [23]:
value_counts_reasons_for_admission = df['patients.reasons_for_admission'].value_counts()
print(value_counts_reasons_for_admission)

patients.reasons_for_admission
cat attack                             609
Cat attack                             434
ss                                     277
HBV                                    260
dog attack                             229
                                      ... 
found laying on back.  cat nearby.       1
found laying on side w/ broken wing      1
found on ground with injured leg         1
Puppy attack                             1
puncture?                                1
Name: count, Length: 12350, dtype: int64


In [24]:
# Set up a regex pattern to match the string "vehicle" or "HBV" or "collision" in the patient.reasons_for_admission column
# vehicle_collision_regex = "vehicle|HBV|HBC|collision|car"
vehicle_collision_regex = r"\bvehicle\b|\bHBV\b|\bHBC\b|\bcollision\b|\bcar\b"
# Find rows where the patients.ressons_for_admission column contains the string "vehicle" or "HBV" or "collision" and examine the results
df[df['patients.reasons_for_admission'].str.contains(vehicle_collision_regex, regex=True, case=False, na=False)]

,patients.address_found,patients.admitted_at,patients.admitted_by,patients.city_found,patients.clinical_signs,patients.common_name,patients.county_found,patients.diagnosis,patients.disposition,patients.found_at,patients.keywords,patients.lat_found,patients.lng_found,patients.name,patients.notes_about_rescue,patients.reason_for_disposition,patients.reasons_for_admission,patients.release_type,species.genus,species.species
0,None,2016-01-01 20:36:00,JB,Middleburg,NaN,Virginia Opossum,Fauquier,"HBV, head trauma",Died in 24hr,2016-01-01,NaN,38.960822,-77.741824,--,NaN,NaN,"helpless in road, limited mobility, appeared t...",NaN,Didelphis,virginiana
38,None,2016-02-06 00:50:00,JA,Fredericksburg,NaN,Great Blue Heron,Spotsylvania County,H- Hit by vehicle,Euthanized +24hr,2016-02-06,NaN,-77.577312,38.270378,--,NaN,no improvement on head trauma - would not surv...,"heron was found on the roadside, likely struck...",NaN,Ardea,herodias
43,None,2016-02-13 22:00:00,JB,Warrenton,NaN,Red-shouldered Hawk,Fauquier County,H - Hit by vehicle,Released,2016-02-13,NaN,38.690077,-77.833481,--,NaN,NaN,Hit by car,Hard,Buteo,lineatus
53,None,2016-02-22 17:30:00,JR/HS,Berryville,NaN,Eastern Cottontail,Clarke County,H- Hit by vehicle,Euthanized in 24hr,2016-02-22,NaN,39.135711,-77.991926,--,NaN,Poor Prognosis,Hit by car,NaN,Sylvilagus,floridanus
55,None,2016-02-24 17:30:00,HS,Bluemont,NaN,Red-shouldered Hawk,Loudoun County,H- Hit by vehicle,Died in 24hr,2016-02-24,NaN,39.111266,-77.835161,--,NaN,NaN,Hit by car,NaN,Buteo,lineatus
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25224,None,2025-02-05 11:11:00,AW,Amissville,NaN,Red-tailed Hawk,Rappahannock,T - Unknown Trauma,Euthanized +24hr,2025-02-05,suspect HBV,38.718336,-78.065992,NaN,NaN,Poor Prognosis,"Suspect HBV, found down on the side of the hig...",NaN,Buteo,jamaicensis
25233,41110 Lovettsville Rd. (nearby),2025-02-07 12:00:00,AW,Lovettsville,NaN,Red-shouldered Hawk,Loudoun,H - Hit by Vehicle,Euthanized in 24hr,2025-02-06,NaN,NaN,NaN,NaN,NaN,Poor Prognosis,"HBV, original finder is unknown, he dropped th...",NaN,Buteo,lineatus
25251,14 Twin lakes Dr.,2025-02-14 16:21:00,AW,Stafford,NaN,Northern Mockingbird,Stafford,T - Unknown Trauma,Pending,2025-02-14,suspect HBV,NaN,NaN,NaN,NaN,NaN,"Susp. HBV, unable to fly",NaN,Mimus,polyglottos
25257,10295 Koontz Corner Rd.,2025-02-15 15:11:00,AW,Broadway,NaN,Common Grackle,Rockingham,T - Unknown Trauma,Euthanized in 24hr,2025-02-15,suspect HBV,NaN,NaN,NaN,NaN,Poor Prognosis,"Suspect HBV, was found in a part of finders pr...",NaN,Quiscalus,quiscula


In [25]:
# df_reasons_for_admission = df[df['patients.reasons_for_admission'].str.contains(vehicle_collision_regex, regex=True, case=False, na=False)]
# reasons_for_admission_vc = df_reasons_for_admission['patients.reasons_for_admission'].value_counts()
# print(reasons_for_admission_vc)

In [26]:
# test_string = "Stray cat attacked bird in yard.  Was able to get bird + placed in carrier"
# # Check if the string above matches the regex pattern: "vehicle|HBV|HBC|collision|car"
# import re
# # re.search(vehicle_collision_regex, test_string, re.IGNORECASE)
# # We don't want to match the word "carrier" in the string above with the regex pattern.
# # We can use word boundaries to match the whole word "car" instead of just the substring "car" in the string above
# vehicle_collision_regex = r"\bvehicle\b|\bHBV\b|\bHBC\b|\bcollision\b|\bcar\b"
# re.search(vehicle_collision_regex, test_string, re.IGNORECASE)

In [27]:
# Grab the indexes of the rows that contain the string "Vehicle" in the patient.diagnosis column
reasonforadmission_column_vehicle = df[df['patients.reasons_for_admission'].str.contains(vehicle_collision_regex, regex=True, case=False, na=False)].index

### Unify hit by vechicle indexes

In [28]:
# keyword_column_hbv

In [29]:
# Unify the indexes of the rows that we suspect are related to vehicle collisions
unified_indexes = keyword_column_hbv.union(diagnosis_column_vehicle).union(reasonforadmission_column_vehicle)

In [30]:
# Create a new DataFrame with only the rows that we suspect are related to vehicle collisions
df_vehicle_collisions = df.loc[unified_indexes]

In [31]:
df_vehicle_collisions.shape

(3064, 20)

### Examine disposition lat and long columns

Examining the disposition_lat and disposition_lng columns often gives the coordination of wildlife center or animal hospitals. Suspect that this column is not meaningful for our goal of identifying hot spots of animal-vehicle conflict

In [35]:
df_examime_location = df[['patients.disposition_lat',
       'patients.disposition_lng', 'patients.disposition_location',
       'patients.disposition_subdivision', 'patients.dispositioned_at',
       'patients.dispositioned_by', 'patients.found_at', 'patients.keywords',
       'patients.lat_found', 'patients.lng_found']]

KeyError: "['patients.disposition_lat', 'patients.disposition_lng', 'patients.disposition_location', 'patients.disposition_subdivision', 'patients.dispositioned_at', 'patients.dispositioned_by'] not in index"

In [34]:
# select rows from df_examiem_location_cols where patients.disposition_lat is not null
df_examime_location[df_examime_location['patients.disposition_lat'].notnull()]

NameError: name 'df_examime_location' is not defined

## Convert Address to Geolocation
For rows where we have an address but no lat or long info

### Filter out rows that already have a lat / long entry

In [36]:
# Find rows where both patients.lat_found and patients.lng_found columns are null
df_geocoding_req = df_vehicle_collisions[df_vehicle_collisions['patients.lat_found'].isnull() & df_vehicle_collisions['patients.lng_found'].isnull()]

In [37]:
df_geocoding_req

,patients.address_found,patients.admitted_at,patients.admitted_by,patients.city_found,patients.clinical_signs,patients.common_name,patients.county_found,patients.diagnosis,patients.disposition,patients.found_at,patients.keywords,patients.lat_found,patients.lng_found,patients.name,patients.notes_about_rescue,patients.reason_for_disposition,patients.reasons_for_admission,patients.release_type,species.genus,species.species
33,761 Old Charlestown Rd,2016-02-04 23:00:00,HS,Stephenson,NaN,Northern Cardinal,NaN,H- Hit by vehicle,Died +24hr,2016-02-04,NaN,NaN,NaN,--,NaN,NaN,Found laying on side of the road,NaN,Cardinalis,cardinalis
618,"17 North, right by Sky Meadows",2016-05-31 20:52:00,JA,Paris,NaN,Groundhog,Fauquier,H- Hit by vehicle,Dead on arrival,2016-05-31,NaN,NaN,NaN,NaN,NaN,Dead on arrival,hit by car,NaN,Marmota,monax
942,340S at 3 miles out of Front Royal,2016-06-27 16:41:00,JR,Front Royal,NaN,Woodland Box Turtle,NaN,H- Hit by vehicle,Died in 24hr,2016-08-03,NaN,NaN,NaN,NaN,NaN,NaN,hit by car,NaN,Terrapene,Carolina
1424,361 Whissens Ridge Road,2016-09-03 21:56:00,JA,ss,NaN,Eastern Gray Squirrel,NaN,B- Baby,Died +24hr,2016-10-01,"Orphan, mother HBV B- Baby",NaN,NaN,NaN,NaN,NaN,mom hit by car,NaN,Sciurus,carolinensis
1674,150-200 yards past RR tracks headed towards Th...,2017-02-04 19:18:00,JB/JA,Marshall,NaN,Eastern Screech Owl,NaN,H- Hit by vehicle,Released,2017-02-04,NaN,NaN,NaN,NaN,NaN,ready for release,Sitting on the side of the road,NaN,Megascops,asio
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25251,14 Twin lakes Dr.,2025-02-14 16:21:00,AW,Stafford,NaN,Northern Mockingbird,Stafford,T - Unknown Trauma,Pending,2025-02-14,suspect HBV,NaN,NaN,NaN,NaN,NaN,"Susp. HBV, unable to fly",NaN,Mimus,polyglottos
25255,13662 Milltown Rd.,2025-02-15 13:09:00,AW,Lovettsville,NaN,Virginia Opossum,Loudoun,T - Unknown Trauma,Euthanized in 24hr,2025-02-15,suspect HBV,NaN,NaN,NaN,Used gloves and a shove to contain in cardboar...,Poor Prognosis,Finder initially Saw lying in the road and ass...,NaN,Didelphis,virginiana
25257,10295 Koontz Corner Rd.,2025-02-15 15:11:00,AW,Broadway,NaN,Common Grackle,Rockingham,T - Unknown Trauma,Euthanized in 24hr,2025-02-15,suspect HBV,NaN,NaN,NaN,NaN,Poor Prognosis,"Suspect HBV, was found in a part of finders pr...",NaN,Quiscalus,quiscula
25259,1011 Everett Ct.,2025-02-16 12:48:00,AW,Fredericksburg,NaN,Canada Goose,Spotsylvania,I - Infectious Disease,Euthanized in 24hr,2025-02-13,HPAI suspect,NaN,NaN,NaN,"Noted on AERO form: Head tilt worsening, BAR e...",Poor Prognosis,"AERO says susp. HBV, describes lethargy, head ...",NaN,Branta,canadensis


### Export the data frame that needs GeoCoding

In [34]:
df_geocoding_req.to_pickle('./datasets/WRMD_2014_to_2025_geocoding_req.pkl')

In [33]:
from geopy.geocoders import GoogleV3
from geopy.exc import GeocoderTimedOut
import time
import os
from concurrent.futures import ThreadPoolExecutor


In [ ]:
GOOGLE_API_KEY = os.environ.get('GOOGLE_API_KEY')
geolocator = GoogleV3(api_key=GOOGLE_API_KEY)

In [ ]:
import pandas as pd
from geopy.geocoders import GoogleV3
from geopy.exc import GeocoderTimedOut
import time
from concurrent.futures import ThreadPoolExecutor

# Initialize geolocator with a custom timeout value (in seconds)
geolocator = Nominatim(user_agent="Geopy Library")

# Cache to store geocoded addresses (address as key, (lat, lng) as value) to avoid repeating 
cache = {}

# Function to geocode an address 
def geocode_address(address, retries=3, delay=2):
    """
    Geocode an address with retry logic and timeout handling.
    Includes caching to avoid re-geocoding the same address.
    
    param address: The address to geocode.
    param retries: Number of retries on timeout (default 3).
    param delay: Delay (in seconds) between retries (default 2).
    return: Tuple (latitude, longitude) or (None, None) if not found.
    """
    if address in cache:
        return cache[address]  # Return cached result
    
    for attempt in range(retries):
        try:
            # Attempt to geocode with a 5-second timeout
            location = geolocator.geocode(address, timeout=5)
            
            if location:
                # Cache the result for future use
                cache[address] = (location.latitude, location.longitude)
                return location.latitude, location.longitude
            else:
                return None, None
        
        except GeocoderTimedOut:
            time.sleep(delay)  # Sleep before retrying
        except Exception as e:
            time.sleep(delay)  # Sleep before retrying
    
    return None, None

# Function to process each row in the DataFrame and update coordinates
def process_row(index, row):
    address = row['patients.address_found']
    if pd.notna(address):  # Only geocode if address is not NaN
        lat, lng = geocode_address(address)
        return index, lat, lng
    return index, None, None

# Function to populate missing latitude and longitude in the DataFrame using multi-threading
def populate_missing_coordinates(df):
    with ThreadPoolExecutor(max_workers=10) as executor:
        results = list(executor.map(lambda row: process_row(*row), df.iterrows()))
        
    for index, lat, lng in results:
        if lat is not None and lng is not None:
            df.at[index, 'patients.lat_found'] = lat
            df.at[index, 'patients.lng_found'] = lng
    return df

# Call the function to update missing latitude and longitude
df = populate_missing_coordinates(df)

# Output the updated DataFrame
df


### Export Rows that don't need Geocoding

In [38]:
# Get the rows that have both latitude and longitude
df_geocoding_not_req = df_vehicle_collisions[df_vehicle_collisions['patients.lat_found'].notnull() & df_vehicle_collisions['patients.lng_found'].notnull()]

In [39]:
df_geocoding_not_req

,patients.address_found,patients.admitted_at,patients.admitted_by,patients.city_found,patients.clinical_signs,patients.common_name,patients.county_found,patients.diagnosis,patients.disposition,patients.found_at,patients.keywords,patients.lat_found,patients.lng_found,patients.name,patients.notes_about_rescue,patients.reason_for_disposition,patients.reasons_for_admission,patients.release_type,species.genus,species.species
0,None,2016-01-01 20:36:00,JB,Middleburg,NaN,Virginia Opossum,Fauquier,"HBV, head trauma",Died in 24hr,2016-01-01,NaN,38.960822,-77.741824,--,NaN,NaN,"helpless in road, limited mobility, appeared t...",NaN,Didelphis,virginiana
2,None,2016-01-06 23:30:00,HS,Harper's Ferry,NaN,Black Vulture,Jefferson County,H- Hit by vehicle,Died in 24hr,2016-01-06,NaN,-77.738882,39.325379,--,NaN,NaN,"Found by the side of the road, not standing",NaN,Coragyps,atratus
8,None,2016-01-14 16:00:00,HS,Lucketts,NaN,Barred Owl,Loudoun County,H- Hit by vehicle,Euthanized in 24hr,2016-01-14,NaN,39.219637,-77.523534,--,used gloves,poor prognosis,Broken leg,NaN,Strix,varia
10,None,2016-01-15 17:50:00,HS,Hedgesville,NaN,Barred Owl,Berkeley County,H- Hit by vehicle,Released,2016-01-15,NaN,39.400742,-78.115627,--,NaN,NaN,In the middle of the road,Hard,Strix,varia
16,None,2016-01-19 19:00:00,HS,Leesburg,NaN,American Robin,Loudoun County,H- Hit by vehicle,Released,2016-01-19,NaN,39.041987,-77.605404,--,NaN,NaN,"Stuck to ground in the middle of the road, cou...",Hard,Turdus,migratorius
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25171,None,2025-01-23 13:10:00,AW,Washington,NaN,Eastern Screech Owl,Washington,H - Hit by Vehicle,Euthanized +24hr,2025-01-23,NaN,38.705223,-78.151589,NaN,NaN,Poor Prognosis,"HBV, finder performed cpr using his mouth and ...",NaN,Megascops,asio
25203,None,2025-02-01 09:18:00,KS,Purcellville,NaN,Northern Raccoon,Loudoun,I - Infectious Disease,Euthanized in 24hr,2025-02-01,"distemper suspect, rabies suspect, HPAI suspect",39.318712,-77.713339,NaN,Finder denies any direct contact / potential r...,Poor Prognosis,"Found in the road, suspect HBV",NaN,Procyon,lotor
25208,None,2025-02-01 16:10:00,KS,Winchester,NaN,Woodland Box Turtle,Frederick,T - Unknown Trauma,Pending,2025-02-01,suspect HBV,39.194749,-78.234773,NaN,NaN,NaN,"Found in the road bleeding, suspect HBV",NaN,Terrapene,Carolina
25224,None,2025-02-05 11:11:00,AW,Amissville,NaN,Red-tailed Hawk,Rappahannock,T - Unknown Trauma,Euthanized +24hr,2025-02-05,suspect HBV,38.718336,-78.065992,NaN,NaN,Poor Prognosis,"Suspect HBV, found down on the side of the hig...",NaN,Buteo,jamaicensis


In [40]:
df_geocoding_not_req.to_pickle('./datasets/WRMD_2014_to_2025_geocoding_not_req.pkl')